# The Train-Test Split Lottery & Cross-Validation Lab

A single train-test split is a gamble: depending on how samples fall, the measured performance can swing wildly. This lab demonstrates the variance of single splits, why we need a three-way split (Train/Val/Test), and the statistical hazard of peeking at the test set.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

## 1. The Train-Test Split Lottery

Evaluate the same model on the same dataset across 5 different random 80-20 splits to see the high variance of single-split evaluation.

In [ ]:
# Generate synthetic binary classification dataset
n_samples = 100
X = np.random.randn(n_samples, 2)
y = (X[:, 0] + X[:, 1] > 0).astype(int)

accuracies = []
print(f"{'Split':<8} {'Train Size':<12} {'Test Size':<12} {'Test Accuracy':<15}")
print("-" * 48)

for i in range(5):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=i
    )
    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    accuracies.append(acc)
    print(f"Split {i+1:<2} {len(X_train):<12} {len(X_test):<12} {acc:<15.1%}")

print(f"\nMean Accuracy: {np.mean(accuracies):.1%}")
print(f"Std Dev:       {np.std(accuracies):.1%}")
print(f"Spread Range:  {min(accuracies):.1%} - {max(accuracies):.1%}")

## 2. The Three-Way Split (Train / Val / Test)

Tune hyperparameters on a dedicated validation set, leaving the final test set sealed until model selection is complete.

In [ ]:
# Split: 60% Train, 20% Val, 20% Test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

C_candidates = [0.001, 0.01, 0.1, 1.0, 10.0]
best_C = None
best_val_acc = -1.0

print(f"{'C':<8} {'Train Acc':<12} {'Val Acc':<12}")
print("-" * 32)
for C in C_candidates:
    m = LogisticRegression(C=C, random_state=42, max_iter=1000).fit(X_train, y_train)
    train_acc = accuracy_score(y_train, m.predict(X_train))
    val_acc = accuracy_score(y_val, m.predict(X_val))
    print(f"{C:<8} {train_acc:<12.1%} {val_acc:<12.1%}")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_C = C

# Retrain on Train + Val, then evaluate on Sealed Test Set
X_train_val = np.vstack([X_train, X_val])
y_train_val = np.concatenate([y_train, y_val])
final_model = LogisticRegression(C=best_C, random_state=42, max_iter=1000).fit(X_train_val, y_train_val)
final_test_acc = accuracy_score(y_test, final_model.predict(X_test))

print(f"\nOptimal C Selected: {best_C} (Val Acc: {best_val_acc:.1%})")
print(f"Final Unbiased Test Accuracy: {final_test_acc:.1%}")

## 3. Data Leakage & Test Set Peeking Hazard

Simulate what happens when you peek at the test set to choose your model versus keeping it strictly sealed.

In [ ]:
# Train 5 models and pick the best test score (cheating)
test_scores = []
for i in range(5):
    m = LogisticRegression(C=10**(-i), random_state=i, max_iter=1000).fit(X_train, y_train)
    test_scores.append(accuracy_score(y_test, m.predict(X_test)))

best_leaked_score = max(test_scores)
print(f"Sealed Honest Test Score:  {final_test_acc:.1%}")
print(f"Leaked / Cheated Best Score: {best_leaked_score:.1%}")
print(f"Inflation via Leakage:       {best_leaked_score - final_test_acc:+.1%}")